# HW4 - Annie Sherwood

Exploratory analysis of the hotel bookings data (`hotels.csv`):
loading and inspection, hotel-type counts, missing values, a summary of
continuous variables, IQR-based outlier replacement, mean imputation and
a before/after kurtosis comparison.

In [1]:
import numpy as np
import pandas as pd

# Show every column / row of the small summary tables we print.
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 120)

## Q1. Load & basic shape

Load `hotels.csv` (it must be in the same folder as this notebook), then
print the column names, the dtype of every column, and the shape. The
shape numbers come from `df.shape`, not hard-coded values.

In [2]:
df = pd.read_csv('hotels.csv')

# 1) Full list of column names
print('Column names:')
print(list(df.columns))

# 2) Data type of every column
print('\nData types:')
print(df.dtypes)

# 3) Number of observations and columns, taken from the DataFrame itself
n_rows, n_cols = df.shape
print(f'\nObservations: {n_rows} Columns: {n_cols}')

Column names:
['hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'company', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'reservation_status', 'reservation_status_date']

Data types:
hotel                                 str
is_canceled                         int64
lead_time                           int64
arrival_date_year                   int64
arrival_date_month                    str
arrival_date_week_number            int64
arrival_date_day_of_month           int64
stays_in_weekend_nights             int64
stays_in_week_nigh

## Q2. Hotel type counts

The hotel-type column is discovered from the data rather than assumed:
among the non-numeric (text) columns, pick the one whose values mention
"hotel" most often. Ties are broken by preferring a column whose name
contains "hotel".

In [3]:
text_cols = [
    col for col in df.columns
    if not pd.api.types.is_numeric_dtype(df[col])
]


def hotel_score(col):
    """Share of a column's values that contain the word 'hotel'."""
    values = df[col].dropna().astype(str)
    share = values.str.contains('hotel', case=False).mean()
    return (share, 'hotel' in col.lower())


hotel_col = max(text_cols, key=hotel_score)
print(f'Hotel-type column: {hotel_col}')

# Frequency table sorted by count, largest first
hotel_counts = (
    df[hotel_col]
    .value_counts()
    .sort_values(ascending=False)
    .rename_axis(hotel_col)
    .reset_index(name='count')
)
print('\nFrequency table:')
print(hotel_counts.to_string(index=False))

# Number of distinct hotel types (single integer)
n_hotel_types = df[hotel_col].nunique()
print(f'\n{n_hotel_types}')

Hotel-type column: hotel

Frequency table:
       hotel  count
  City Hotel  79330
Resort Hotel  40060

2


## Q3. Missing-value counts

A single chained expression of the form `dataframe.method1().method2()`:
`isnull()` flags each missing cell as `True`, and `sum()` adds up the
`True` values in each column.

In [4]:
missing_counts = df.isnull().sum()
print(missing_counts)

hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               4
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                           0
agent                              16340
company         

## Q4. Continuous-variable summary table

**Definition used:** a continuous variable is any numeric (int or float)
column with **more than 12** distinct non-missing values. (This follows
the written question. The rubric says "more than 10"; for this dataset
both thresholds give the same list, which the cell below checks.)

Kurtosis uses pandas' `.kurt()`, which is Fisher's (excess) kurtosis, so a
normal distribution has kurtosis 0.

In [5]:
DISTINCT_THRESHOLD = 12

numeric_cols = df.select_dtypes(include='number').columns

# nunique() ignores missing values by default (dropna=True)
continuous_vars = [
    col for col in numeric_cols
    if df[col].nunique() > DISTINCT_THRESHOLD
]
print('Continuous variables:')
print(continuous_vars)

# Check that the rubric's "> 10" threshold gives the same list
rubric_vars = [col for col in numeric_cols if df[col].nunique() > 10]
print('\nSame list with a "> 10" threshold:', rubric_vars == continuous_vars)

Continuous variables:
['lead_time', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes', 'agent', 'company', 'days_in_waiting_list', 'adr']

Same list with a "> 10" threshold: True


In [6]:
summary_table = pd.DataFrame({
    'mean': df[continuous_vars].mean(),
    'std': df[continuous_vars].std(),
    'skew': df[continuous_vars].skew(),
    'kurtosis': df[continuous_vars].kurt(),  # Fisher: normal = 0
})
summary_table.index.name = 'variable'
summary_table.round(4)

,mean,std,skew,kurtosis
variable,,,,
lead_time,104.0114,106.8631,1.3465,1.6964
arrival_date_week_number,27.1652,13.6051,-0.0100,-0.9861
arrival_date_day_of_month,15.7982,8.7808,-0.0020,-1.1872
stays_in_weekend_nights,0.9276,0.9986,1.3800,7.1741
stays_in_week_nights,2.5003,1.9083,2.8622,24.2846
adults,1.8564,0.5793,18.3178,1352.1151
previous_cancellations,0.0871,0.8443,24.4580,674.0737
previous_bookings_not_canceled,0.1371,1.4974,23.5398,767.2452
booking_changes,0.2211,0.6523,6.0003,79.3936


## Q5. Outlier replacement

**Rule:** for each continuous variable, compute Q1, Q3 and
IQR = Q3 - Q1 by hand with `quantile()`. A value is an outlier if it is
below Q1 - 1.8 × IQR or above Q3 + 1.8 × IQR. Outliers are replaced with
`np.nan`.

The original `df` is left untouched; changes go into a copy, `df_clean`,
so we can still compare against the original data in Q7.

In [7]:
IQR_MULTIPLIER = 1.8

df_clean = df.copy()
outlier_counts = {}

for col in continuous_vars:
    q1 = df_clean[col].quantile(0.25)
    q3 = df_clean[col].quantile(0.75)
    iqr = q3 - q1
    lower_fence = q1 - IQR_MULTIPLIER * iqr
    upper_fence = q3 + IQR_MULTIPLIER * iqr

    # Comparisons with NaN are False, so missing values never count
    is_outlier = (
        (df_clean[col] < lower_fence) | (df_clean[col] > upper_fence)
    )
    outlier_counts[col] = int(is_outlier.sum())
    df_clean.loc[is_outlier, col] = np.nan

# Keep only the variables that actually had outliers
outlier_table = (
    pd.Series(outlier_counts, name='outliers_replaced')
    .loc[lambda s: s > 0]
    .rename_axis('variable')
    .reset_index()
)
print(outlier_table.to_string(index=False))

outlier_vars = outlier_table['variable'].tolist()

                      variable  outliers_replaced
                     lead_time               1741
       stays_in_weekend_nights                265
          stays_in_week_nights               3354
                        adults              29710
        previous_cancellations               6484
previous_bookings_not_canceled               3620
               booking_changes              18076
          days_in_waiting_list               3698
                           adr               2524


## Q6. Mean imputation

Only the variables listed in the Q5 table are imputed. Each one is filled
with its mean *after* outlier removal (pandas `mean()` skips `NaN`, so the
mean comes from the values left after Q5). This fills both the new `NaN`s
from Q5 and any values that were already missing in those columns. No
other columns are changed.

In [8]:
post_removal_means = df_clean[outlier_vars].mean()
df_clean[outlier_vars] = df_clean[outlier_vars].fillna(post_removal_means)

# Confirm that every imputed variable now has zero missing values
remaining_missing = df_clean[outlier_vars].isnull().sum()
print('Missing values after imputation:')
print(remaining_missing)
all_complete = bool((remaining_missing == 0).all())
print('\nAll imputed variables complete:', all_complete)

Missing values after imputation:
lead_time                         0
stays_in_weekend_nights           0
stays_in_week_nights              0
adults                            0
previous_cancellations            0
previous_bookings_not_canceled    0
booking_changes                   0
days_in_waiting_list              0
adr                               0
dtype: int64

All imputed variables complete: True


## Q7. Kurtosis comparison (before vs after)

For each variable changed in Q5/Q6, compare kurtosis before any outlier
treatment (from the Q4 table) with kurtosis after outlier removal and
mean imputation. The table is sorted by the absolute change, largest
first.

In [9]:
kurtosis_before = summary_table.loc[outlier_vars, 'kurtosis']
kurtosis_after = df_clean[outlier_vars].kurt()

kurtosis_comparison = pd.DataFrame({
    'kurtosis_before': kurtosis_before,
    'kurtosis_after': kurtosis_after,
    'difference': kurtosis_after - kurtosis_before,
})
kurtosis_comparison = kurtosis_comparison.sort_values(
    by='difference', key=np.abs, ascending=False
)
kurtosis_comparison.index.name = 'variable'
kurtosis_comparison.round(4)

,kurtosis_before,kurtosis_after,difference
variable,,,
adults,1352.1151,0.0000,-1352.1151
adr,1013.1899,0.3503,-1012.8396
previous_bookings_not_canceled,767.2452,0.0000,-767.2452
previous_cancellations,674.0737,0.0000,-674.0737
days_in_waiting_list,186.7931,0.0000,-186.7931
booking_changes,79.3936,0.0000,-79.3936
stays_in_week_nights,24.2846,-0.2770,-24.5616
stays_in_weekend_nights,7.1741,0.0717,-7.1023
lead_time,1.6964,0.3806,-1.3159


In [10]:
# The variable with the largest change, for the conclusion below
top_var = kurtosis_comparison.index[0]
top_row = kurtosis_comparison.iloc[0]
direction = 'less' if top_row['difference'] < 0 else 'more'
print(
    f"{top_var}: kurtosis went from {top_row['kurtosis_before']:.2f} "
    f"to {top_row['kurtosis_after']:.2f} "
    f"(change {top_row['difference']:.2f}), so it is now "
    f"{direction} heavy-tailed."
)

adults: kurtosis went from 1352.12 to 0.00 (change -1352.12), so it is now less heavy-tailed.


**Conclusion:** `adults` had the largest kurtosis change, dropping from about
1352 to 0 (a change of about -1352), so its distribution became far
**less** heavy-tailed. Its Q1 and Q3 are both 2, so the IQR is 0 and every
value other than 2 counted as an outlier. After mean imputation the column
is constant, so pandas reports its kurtosis as 0. The other modified
variables also have lower kurtosis after treatment (every difference is
negative). This means trimming at 1.8 × IQR and filling with the mean
removed the heavy tails in every case.